# Day 5 Tutorial - Oxford Socratic (SQL与数据治理)

## Persona

You are an Oxford tutorial fellow in **SQL与数据治理** (sqlite3 + pandas.read_sql + 六维数据治理审计 + RFM 52/49/31/45).

**Rules**:
- Never give direct answers. 不直接给答案, 不直接答.
- Use Socratic questioning - 反例, 凭什么, 为什么, 若...变.
- Act as devil's advocate - reject vague claims, demand precision.
- End each turn with a probing question.
- Detect defense failure -> drop one scaffold level, still no direct answer.

## Topic

Day 5 数据治理与 SQL - 电商营销数据库 Schema 设计, SQL DQL 查询, RFM 52/49/31/45 分群, 六维数据治理审计 (DAMA-DMBOK).

## Pre-tutorial Task (强制 retrieval - 学生先提交)

本 tutorial 开始前, 学生必须独立完成 (强制提取练习 retrieval practice):

1. 用 sqlite3 创建电商营销数据库 6 表 Schema (customers/products/categories/orders/order_items/campaigns), 含 PRIMARY KEY / FOREIGN KEY / CHECK / DEFAULT 四类约束.
2. 写 SQL 查询: '某客户买了什么产品' (3 表 JOIN) + 按品类聚合 GMV (GROUP BY-HAVING).
3. 写 RFM 52/49/31/45 分群 SQL 思路 (用窗口函数 RANK + CTE).
4. 列出数据治理六维度, 各写一条 SQL 检测语句.

未提交 pre-task 不得进入 tutorial - 牛津 tutorial 假设学生已做完尝试.

In [ ]:
# Socratic tutorial loop - 4+ turns, static if/else simulation (no API call)
# 每轮: 检测 defense 失败 -> 降一级 scaffold, 仍禁直接答案

student_responses = {
    "turn1": "我用 SELECT * FROM customers 查询所有客户",
    "turn2": "我用 JOIN 连接 customers 和 orders",
    "turn3": "RFM 就是按 R F M 三个维度分群, 阈值用 5 和 2",
    "turn4": "数据质量就是看数据准不准",
}

def socratic_turn1(resp):
    # 为什么用 SELECT *? 若新增 PII 列会泄露什么? 反例: GDPR 合规吗? 凭什么?
    if "SELECT *" in resp:
        return ("为什么用 SELECT * 而不是显式列名? 若 customers 表新增一列 PII (手机号), "
                "SELECT * 会泄露什么? 反例: 在 GDPR / 个保法下这条查询合规吗? "
                "凭什么认为 SELECT * 是好实践? 如何用 pandas.read_sql 只取需要的列?")
    return "如何用 SQL 表达'某客户买了什么产品'? 需要哪几张表 JOIN?"

def socratic_turn2(resp):
    # 若 JOIN 缺 ON -> 笛卡尔积多大? 假设表行数变了怎么验证?
    if "JOIN" in resp and "ON" not in resp:
        return ("若 JOIN 缺 ON 会发生什么? 笛卡尔积多大? "
                "如何用 pandas.read_sql 验证返回 DataFrame 的 shape? "
                "假设 customers 200 行 orders 500 行, 缺 ON 返回多少行? "
                "凭什么认为 JOIN 不需要 ON? 反例: 写一条缺 ON 的 SQL 看结果.")
    return "JOIN 的 ON 条件是什么? 为什么不能省?"

def socratic_turn3(resp):
    # RFM 52/49/31/45 阈值从哪来? 凭什么用 5 和 2? 假设业务变了怎么调? 反例?
    if "RFM" in resp and ("5" in resp or "2" in resp):
        return ("RFM 52/49/31/45 的阈值从哪来? 凭什么用 5 和 2? "
                "是业务定还是 pandas.describe 分位数定? 假设业务变了阈值怎么调? "
                "反例: 若所有客户 R 都很高, 52 分群还有意义吗? "
                "如何用窗口函数 RANK 和 DENSE_RANK 计算 R/F/M? 二者并列时有何差异?")
    return "如何用 SQL 窗口函数计算 R/F/M 排名? RANK 和 DENSE_RANK 有何差异?"

def socratic_turn4(resp):
    # 六维漏了哪 5 维? 如何用 SQL 检测? 反例: 准确但不及时? 凭什么?
    if "准不准" in resp or "准确" in resp:
        return ("数据治理六维度 (准确性/完整性/一致性/及时性/唯一性/有效性) 你只提了准确性, "
                "漏了哪 5 维? 如何用 SQL COUNT(DISTINCT) / IS NULL / strftime 各写一条检测? "
                "反例: 准确但不及时的数据算合格吗? 凭什么? "
                "假设业务变了数据生命周期, 你的检测规则要怎么更新?")
    return "数据质量六维度是哪六个? 各用什么 SQL 检测?"

# Run 4-turn Socratic loop
turn = 1
for resp_key, resp_val in student_responses.items():
    if turn == 1:
        feedback = socratic_turn1(resp_val)
    elif turn == 2:
        feedback = socratic_turn2(resp_val)
    elif turn == 3:
        feedback = socratic_turn3(resp_val)
    else:
        feedback = socratic_turn4(resp_val)
    print(f"=== Turn {turn} (scaffold level: {5 - turn}) ===")
    print(f"Student: {resp_val}")
    print(f"Fellow:  {feedback}")
    print()
    turn += 1

print("Tutorial 4 轮完成. 仍禁直接答案 - 学生须自己重做 drill D1/D2/D3 阶段3.")

In [ ]:
# student_model.json - 跨单元复用的学生掌握度模型
# 记录盲点 / 掌握度 / Hattie 反馈历史, 下一单元 tutorial 可读取
import json
import os

student_model = {
    "unit": "U0D5",
    "topic": "数据治理与SQL",
    "mastered_subskills": ["S1.partial", "S2.partial"],
    "weak_points": [
        "sqlite3 DDL 约束设计 (FOREIGN KEY 缺失)",
        "RFM 窗口函数 RANK vs DENSE_RANK",
        "六维审计漏维度 (只提准确性)",
    ],
    "socratic_turns_passed": [1, 2],
    "socratic_turns_failed": [3, 4],
    "hattie_feedback_history": [
        {"turn": 1, "level": "TASK", "issue": "SELECT * 不指定列 - PII 泄露风险"},
        {"turn": 2, "level": "PROCESS", "issue": "JOIN 缺 ON - 笛卡尔积"},
        {"turn": 3, "level": "SELF-REG", "issue": "RFM 阈值未反思 - 业务 vs 分位数"},
        {"turn": 4, "level": "FEED-FORWARD", "issue": "六维审计漏维度 -> 回 drill D3"},
    ],
    "review_units": ["U0D5", "U0D4"],
    "next_due_cards": ["C1", "C3"],  # schedule.json 中的 card id
    "last_updated": "2026-07-25",
}

# 读写 student_model.json - 跨单元复用
model_path = "student_model.json"
if os.path.exists(model_path):
    with open(model_path, encoding="utf-8") as f:
        existing = json.load(f)
    # 合并 - 新数据覆盖旧数据
    existing.update(student_model)
    student_model = existing

with open(model_path, "w", encoding="utf-8") as f:
    json.dump(student_model, f, ensure_ascii=False, indent=2)

print(f"student_model.json 已更新")
print(f"盲点数: {len(student_model['weak_points'])}")
print(f"建议复习单元: {student_model['review_units']}")
print(f"明天的 due cards: {student_model['next_due_cards']}")

## Hattie 四级形成性反馈 (避免 Self 级表扬)

> Hattie (2007 RER 77(1):81-112) 3 问 × 4 级 - 不表扬学生本人 (Self 级), 而是反馈任务/过程/自我调节/前向.

基于刚才 4 轮 Socratic 对话, 给学生的反馈:

- **[TASK]** 任务级反馈: 你的 sqlite3 DDL 缺 FOREIGN KEY - orders 表的 customer_id 没有引用约束, 会产生孤儿订单. 重写 DDL 加 `FOREIGN KEY (customer_id) REFERENCES customers(customer_id)`. (反馈任务本身的对错)

- **[PROCESS]** 过程级反馈: 你用 SELECT * 然后用 pandas 过滤 - 更高效的做法是用 SQL WHERE 预过滤再 `pandas.read_sql`, 减少 Python 内存. SQL 还是 pandas 处理取决于数据量, 这是过程选择问题. (反馈解题策略)

- **[SELF-REG]** 自我调节级反馈: 你跳过了 worked-faded 阶段2 直接做阶段3 - 自评: 你能解释 RANK 和 DENSE_RANK 在并列时的差异吗? 若不能, 回阶段2 复看 worked example. (反馈学生的自我监控)

- **[FEED-FORWARD]** 前向级反馈: 基于你的 student_model.json (盲点: RFM 窗口函数 + 六维审计漏维度), 下一单元 Day 6 研究方法论会复用窗口函数做时间序列分析 - 提前复习 Day 5 drill D2/D3 阶段3. (反馈下一步学习路径)

## 限频与 Exit (防依赖)

**限频 (daily limit)**: 每单元 tutorial **每天 1 次** (1次/天), 防止学生把 Socratic tutor 当答案机. 超过则第二天再来 - 强制间隔重复 (spaced retrieval), 让大脑有时间巩固.

**Exit artifact** (本单元 tutorial 结束时学生须提交):

1. **2-3 个本单元盲点** (从 student_model.json 的 weak_points 中选): 例如 sqlite3 DDL 约束 / RFM 窗口函数 / 六维审计漏维度.
2. **推荐复习单元** (基于 review_units): U0D5 (重做 drill D3) + U0D4 (统计基础复用).
3. **100 字自我反思**: 我在 sqlite3/SQL/RFM/数据治理 上原来的理解 vs tutorial 后的理解差异.
4. **下一阶段计划**: 哪个 drill 阶段3 要重做? 哪个 schedule.json card 明天 due?

未经 Exit 不得进入下一单元 - tutorial 不是聊天, 是建构对齐 (constructive alignment) 的 TLA. Exit artifact 是 AT (评估任务).